# microgrid-dispatch-milp — full experiment run (Colab, CPU only)

Use **Runtime → Run all** (or run the cells in order). Any failed check raises an error, which stops Run all. No GPU is needed (Runtime → Change runtime type → CPU).

* All results and checkpoints are written to Google Drive (`MyDrive/microgrid/results`).
* If the session disconnects: reconnect, then run cells 1, 2 and 5 again. Finished runs are skipped and unfinished runs resume from their last weekly checkpoint.
* Keep this browser tab open while it runs (free Colab stops idle sessions).

In [ ]:
# 1) Mount Google Drive (a pop-up asks for permission)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Unpack the code from Drive and install dependencies (~1-2 min)
import os, subprocess, pathlib
DRIVE = '/content/drive/MyDrive/microgrid'
ZIP = f'{DRIVE}/microgrid-dispatch-milp.zip'
assert os.path.exists(ZIP), f'Upload microgrid-dispatch-milp.zip to {DRIVE} first'
!rm -rf /content/mg && mkdir -p /content/mg && unzip -q "$ZIP" -d /content/mg
CODE = '/content/mg/microgrid-dispatch-milp'
!pip install -q -r {CODE}/requirements.txt
!pip install -q gurobipy cplex 2>/dev/null | tail -1
!apt-get -qq install -y coinor-cbc > /dev/null 2>&1 || echo 'CBC not installed (optional)'
RESULTS = f'{DRIVE}/results'
LOGS = f'{DRIVE}/logs'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
os.environ['MG_RESULTS_DIR'] = RESULTS
print('Code:', CODE, '\nResults ->', RESULTS)

In [ ]:
# 3) Unit tests (~1 min). Raises an error (stops 'Run all') if any test fails.
import subprocess
r = subprocess.run(f'cd {CODE} && python -m pytest -q 2>&1 | tail -3', shell=True, capture_output=True, text=True)
print(r.stdout)
assert 'passed' in r.stdout and 'failed' not in r.stdout and 'error' not in r.stdout.lower(), 'Unit tests failed - send this output to Claude'

In [ ]:
# 4) Smoke test (~3 min, temporary folder, NOT Drive). Raises an error if anything fails.
r = subprocess.run(f'cd {CODE} && MG_RESULTS_DIR=/content/quick_results python experiments/run_all.py --quick --passes 1,2 2>&1 | tail -4',
                   shell=True, capture_output=True, text=True)
print(r.stdout)
assert 'failures: 0' in r.stdout, 'Smoke test failed - send this output to Claude'

In [ ]:
# 5) Start the full run: 2 background processes (one per CPU core).
#    Safe to re-run: it does nothing if the processes are already running.
import subprocess, os
running = subprocess.run(['pgrep', '-f', 'experiments/run_all.py'], capture_output=True, text=True).stdout.split()
if running:
    print('Already running (PIDs', running, ') - go to cell 6.')
else:
    for i in range(2):
        cmd = (f'cd {CODE} && OMP_NUM_THREADS=1 MG_RESULTS_DIR="{RESULTS}" '
               f'nohup python experiments/run_all.py --shard {i}/2 >> "{LOGS}/shard{i}.log" 2>&1 &')
        subprocess.run(cmd, shell=True)
    print('Started 2 shards. Logs:', LOGS)

In [ ]:
# 6) Progress monitor. It prints every 5 minutes; you may stop this cell at any time
#    (the background runs continue). Expected total: 468 result files.
import time, glob, subprocess
TOTAL = 468
for _ in range(12 * 12):                       # up to 12 hours
    done = len([f for f in glob.glob(f'{RESULTS}/0[2345]_*/raw/*/*.json')])
    alive = subprocess.run(['pgrep', '-f', 'experiments/run_all.py'], capture_output=True, text=True).stdout.split()
    print(time.strftime('%H:%M'), f'finished runs: {done}/{TOTAL}', '| processes running:', len(alive))
    for i in range(2):
        print(f'  shard{i}:', subprocess.run(['tail', '-n', '1', f'{LOGS}/shard{i}.log'], capture_output=True, text=True).stdout.strip()[:150])
    if not alive:
        break
    time.sleep(300)
if done < TOTAL:
    raise RuntimeError(f'Only {done}/{TOTAL} runs finished (session limit or a crash). '
                       'Re-run cells 1, 2, 5 and 6 - finished runs are skipped. '
                       'If the log shows FAILED, send the logs to Claude.')
print('All runs finished.')

In [ ]:
# 7) Short experiments 06-07 (~10 min) and the summary. Refuses to run on partial results.
done = len(glob.glob(f'{RESULTS}/0[2345]_*/raw/*/*.json'))
assert done >= 468, f'Only {done}/468 runs finished - run cells 5 and 6 first'
!cd {CODE} && python experiments/06_milp_vs_lp.py 2>&1 | tail -5
!cd {CODE} && python experiments/07_solvers.py 2>&1 | grep -v 'size restricted' | tail -12
!cd {CODE} && python experiments/08_summarize.py 2>&1 | tail -3
!grep -h -c 'FAILED' {LOGS}/shard*.log || true

In [ ]:
# 8) Pack the results (without checkpoints) for download / sending to Claude
!cd "{DRIVE}" && rm -f results_for_claude.zip && zip -qr results_for_claude.zip results logs -x '*.ckpt.npz' '*.tmp.npz'
!ls -lh "{DRIVE}/results_for_claude.zip"
from google.colab import files
files.download(f'{DRIVE}/results_for_claude.zip')